# F1 Telemetry Embedding Generation with Chronos-2
This notebook downloads the F1 demo dataset from Hugging Face, generates multivariate time-series embeddings using Amazon's Chronos-2 model, and visualizes the results with Renumics Spotlight.

## Download dataset from Hugging Face

In [8]:
import datasets
import pandas as pd
import numpy as np
from renumics.spotlight.dtypes import Sequence1D

ds = datasets.load_dataset("renumics/f1_demo_dataset", split="train")
df = ds.to_pandas()

sequence_cols = ["speed", "throttle", "nGear", "brake", "drs", "x", "y", "z", "distance_driver"]
for col in sequence_cols:
    df[col] = df[col].apply(lambda s: Sequence1D(index=s[0], value=s[1]) if s is not None and len(s) == 2 else None)

df.info()
df.head()

Gtk-Message: 14:01:38.171: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 51 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   Time                201 non-null    timedelta64[ns]
 1   Driver              201 non-null    object         
 2   DriverNumber        201 non-null    object         
 3   LapTime             201 non-null    float64        
 4   LapNumber           201 non-null    float64        
 5   Stint               201 non-null    float64        
 6   PitOutTime          8 non-null      timedelta64[ns]
 7   PitInTime           5 non-null      timedelta64[ns]
 8   Sector1Time         198 non-null    float64        
 9   Sector2Time         201 non-null    float64        
 10  Sector3Time         201 non-null    float64        
 11  Sector1SessionTime  198 non-null    timedelta64[ns]
 12  Sector2SessionTime  201 non-null    timedelta64[ns]
 13  Sector3SessionTime  201 non-null   

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,brake_emb,throttle_emb,x_emb,y_emb,z_emb,gear_vis,speed_vis,portrait,brake_emb_reduced,__index_level_0__
0,0 days 01:03:28.836000,VER,1,83.080,1.0,1.0,0 days 00:15:58.684000,NaT,NaN,25.105,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[16.0, 16.0, 16.0, 24.03174223906167, 38.85620...",NaN,NaN,NaN,imgs/gears/gear_shift_vis_0.png,imgs/speed/speed_vis_0.png,imgs/driver/VER.jpg,"[11.521774291992188, -2.314301013946533]",0
1,0 days 01:04:46.911000,VER,1,78.075,2.0,1.0,NaT,NaT,21.686,25.051,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100...",NaN,NaN,NaN,imgs/gears/gear_shift_vis_1.png,imgs/speed/speed_vis_1.png,imgs/driver/VER.jpg,"[9.902015686035156, -1.8040274381637573]",1
2,0 days 01:06:04.678000,VER,1,77.767,3.0,1.0,NaT,NaT,21.626,24.909,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100...",NaN,NaN,NaN,imgs/gears/gear_shift_vis_2.png,imgs/speed/speed_vis_2.png,imgs/driver/VER.jpg,"[10.624198913574219, -3.003828763961792]",2
3,0 days 01:07:21.881000,VER,1,77.203,4.0,1.0,NaT,NaT,21.438,24.716,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100...",NaN,NaN,NaN,imgs/gears/gear_shift_vis_3.png,imgs/speed/speed_vis_3.png,imgs/driver/VER.jpg,"[10.201083183288574, -1.0347868204116821]",3
4,0 days 01:08:39.252000,VER,1,77.371,5.0,1.0,NaT,NaT,21.493,24.718,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100...",NaN,NaN,NaN,imgs/gears/gear_shift_vis_4.png,imgs/speed/speed_vis_4.png,imgs/driver/VER.jpg,"[10.323925971984863, -2.0249626636505127]",4


## Visualize dataset with Spotlight

In [ ]:
from renumics import spotlight

sequence_cols = ["speed", "throttle", "nGear", "brake", "drs", "x", "y", "z", "distance_driver"]
dtype = {c: spotlight.dtypes.sequence_1d_dtype for c in sequence_cols}


spotlight.show(ds, dtype=dtype, port=8890, no_ssl=True, host="0.0.0.0", analyze=False)

Gtk-Message: 14:01:41.790: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.


## Generate embeddings with Chronos-2
We use the multivariate embedding capability of Chronos-2 to embed speed, throttle, and nGear signals jointly. Cross-variate attention captures relationships between the signals.

In [3]:
import torch
from chronos import Chronos2Pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device)

Gtk-Message: 14:01:22.001: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
/home/stefan/code/tutorial-testing-data-analysis/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [4]:
def extract_signal_values(series_col):
    """Extract y-values from Sequence1D objects."""
    signals = []
    for row in series_col:
        if row is not None and hasattr(row, "value"):
            signals.append(np.array(row.value, dtype=np.float32))
        else:
            signals.append(np.array([], dtype=np.float32))
    return signals


speed_signals = extract_signal_values(df["speed"])
throttle_signals = extract_signal_values(df["throttle"])
ngear_signals = extract_signal_values(df["nGear"])

max_len = max(
    max(len(s) for s in speed_signals),
    max(len(s) for s in throttle_signals),
    max(len(s) for s in ngear_signals),
)
print(f"Max signal length: {max_len}")

Max signal length: 1162


In [5]:
def pad_signal(signal, target_len):
    """Right-pad signal with last value to target length."""
    if len(signal) == 0:
        return np.zeros(target_len, dtype=np.float32)
    if len(signal) >= target_len:
        return signal[:target_len]
    return np.pad(signal, (0, target_len - len(signal)), mode="edge")


batch_size = 8
all_embeddings = []

for i in range(0, len(df), batch_size):
    batch_tensors = []
    for j in range(i, min(i + batch_size, len(df))):
        speed_padded = pad_signal(speed_signals[j], max_len)
        throttle_padded = pad_signal(throttle_signals[j], max_len)
        ngear_padded = pad_signal(ngear_signals[j], max_len)
        stacked = np.stack([speed_padded, throttle_padded, ngear_padded])  # (3, max_len)
        batch_tensors.append(stacked)

    batch = torch.tensor(np.stack(batch_tensors), dtype=torch.float32)  # (B, 3, max_len)

    with torch.no_grad():
        embedding_list, _ = model.embed(batch)
        pooled = torch.stack([emb.mean(dim=(0, 1)) for emb in embedding_list])

    all_embeddings.append(pooled.cpu().numpy())
    print(f"Processed {min(i + batch_size, len(df))}/{len(df)} samples")

embeddings_array = np.concatenate(all_embeddings, axis=0)
print(f"Final embeddings shape: {embeddings_array.shape}")

Processed 8/201 samples
Processed 16/201 samples
Processed 24/201 samples
Processed 32/201 samples
Processed 40/201 samples
Processed 48/201 samples
Processed 56/201 samples
Processed 64/201 samples
Processed 72/201 samples
Processed 80/201 samples
Processed 88/201 samples
Processed 96/201 samples
Processed 104/201 samples
Processed 112/201 samples
Processed 120/201 samples
Processed 128/201 samples
Processed 136/201 samples
Processed 144/201 samples
Processed 152/201 samples
Processed 160/201 samples
Processed 168/201 samples
Processed 176/201 samples
Processed 184/201 samples
Processed 192/201 samples
Processed 200/201 samples
Processed 201/201 samples
Final embeddings shape: (201, 768)


In [11]:
if "chronos_embedding" in ds.column_names:
    ds = ds.remove_columns("chronos_embedding")
ds = ds.add_column("chronos_embedding", [emb.tolist() for emb in embeddings_array])
print(f"Added chronos_embedding to ds: {ds.features['chronos_embedding']}")
print(f"Sample embedding length: {len(ds[0]['chronos_embedding'])}")

Added chronos_embedding to ds: List(Value('float64'))
Sample embedding length: 768


## Visualize embeddings with Spotlight

In [12]:
from renumics import spotlight

sequence_cols = ["speed", "throttle", "nGear", "brake", "drs", "x", "y", "z", "distance_driver"]
dtype = {c: spotlight.dtypes.sequence_1d_dtype for c in sequence_cols}
for c in ["portrait", "speed_vis", "gear_vis"]:
    dtype[c] = spotlight.dtypes.image_dtype
embedding_cols = [
    "chronos_embedding", "speed_emb", "brake_emb", "throttle_emb", "nGear_emb",
    "brake_emb_reduced", "speed_emb_reduced", "throttle_emb_reduced",
    "nGear_emb_reduced", "chronos_embedding_reduced",
]
for c in embedding_cols:
    if c in ds.column_names:
        dtype[c] = spotlight.dtypes.embedding_dtype

spotlight.show(ds, dtype=dtype, port=8891, no_ssl=True, host="0.0.0.0", analyze=False)

Gtk-Message: 14:07:31.424: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.


## Embedding size reduction
Use UMAP to reduce high-dimensional embeddings to 2D for visualization in Spotlight's similarity map.

In [ ]:
import umap

embedding_cols = {
    "speed_emb": ds["speed_emb"],
    "throttle_emb": ds["throttle_emb"],
    "chronos_embedding": ds["chronos_embedding"],
}

# For nGear, pad signal values to uniform length as a flat embedding
ngear_raw = ds["nGear"]
ngear_max_len = max(len(row[1]) for row in ngear_raw)
ngear_padded = []
for row in ngear_raw:
    vals = np.array(row[1], dtype=np.float32)
    if len(vals) < ngear_max_len:
        vals = np.pad(vals, (0, ngear_max_len - len(vals)), mode="edge")
    ngear_padded.append(vals.tolist())
embedding_cols["nGear_emb"] = ngear_padded

for col_name, col_data in embedding_cols.items():
    arr = np.array(col_data, dtype=np.float32)
    reducer = umap.UMAP(n_components=2, random_state=42)
    reduced = reducer.fit_transform(arr)

    reduced_col = f"{col_name}_reduced"
    if reduced_col in ds.column_names:
        ds = ds.remove_columns(reduced_col)
    ds = ds.add_column(reduced_col, reduced.tolist())
    print(f"{col_name} ({arr.shape[1]}d) -> {reduced_col} (2d)")

print(f"\nDataset columns: {ds.column_names}")